In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
import gc

In [2]:
gc.collect()
df1 = pd.read_csv("./df_knni_rounded.csv", sep='\t').copy().reset_index(drop = True)
df1

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.230,26.34,72.93,3.60246,...,401.0,0.0,12.0,24.0,1.0,2.0,1.0,2.0,1.0,2.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.120,44.83,86.71,3.36580,...,1094.0,0.0,0.0,49.0,8.0,3.0,4.0,0.0,1.0,3.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,40.086,33.65,72.17,3.41520,...,645.0,0.0,17.0,48.0,3.0,2.0,4.0,2.0,0.0,6.0
3,2022.0,47.35,83700.0,6.44,353.0,85.11,54.320,37.73,82.88,3.42020,...,867.0,0.0,18.0,18.0,1.0,3.0,4.0,0.0,1.0,1.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.730,65.29,86.17,3.45560,...,1090.0,0.0,16.0,29.0,0.0,0.0,4.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,2016.0,45.99,182311.0,10.11,817.0,86.91,30.510,42.42,84.85,2.89010,...,2691.0,0.0,2.0,40.0,1.0,2.0,0.0,1.0,1.0,1.0
199996,2015.0,67.65,275317.0,11.06,840.0,65.08,77.610,30.76,65.08,3.57070,...,420.0,0.0,8.0,0.0,4.0,3.0,4.0,0.0,1.0,0.0
199997,2019.0,98.08,158747.0,4.89,336.0,86.08,44.590,24.64,84.90,2.98750,...,3148.0,0.0,2.0,40.0,0.0,4.0,0.0,1.0,0.0,2.0
199998,2016.0,51.49,95122.0,10.06,392.0,76.61,80.700,41.93,75.50,3.56980,...,797.0,0.0,16.0,30.0,7.0,0.0,4.0,1.0,1.0,1.0


In [26]:
class LinearSVR_Class(object):
    def __init__(self, dataset, epsilon : float, tol : float, C : float, 
                 loss : str, fit_intercept : bool, 
                 intercept_scaling : float, dual : 'auto' or bool, verbose : int, 
                 random_state : int or None, max_iter : int):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.epsilon = epsilon
        self.tol = tol
        self.C = C
        self.loss = loss
        self.fit_intercept = fit_intercept
        self.intercept_scaling = intercept_scaling
        self.dual = dual
        self.verbose = verbose
        self.random_state = random_state
        self.max_iter = max_iter
        
    def train_split(self):

        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        from sklearn.model_selection import train_test_split
        df4 = self.dataset.copy().reset_index(drop = True)
        Y_col = 'Fleet_Or_Private_Num_Value'
        X_cols = df4.loc[:, df4.columns != Y_col].columns
        # Split into 80% training and 20% testing
        X_train, X_test, y_train, y_test = train_test_split(
            df4[X_cols], df4[Y_col], test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train.round()
        y_test_global = y_test.round()

    def ML_fitting(self):
        global gcp_fit_global

        
        gcp = make_pipeline(StandardScaler(), LinearSVR(epsilon=self.epsilon, tol=self.tol, C=self.C, 
                                                        loss=self.loss, fit_intercept=self.fit_intercept, 
                                                        intercept_scaling=self.intercept_scaling, dual=self.dual, 
                                                        verbose=self.verbose, 
                                                        random_state=self.random_state, max_iter=self.max_iter))
        gcp_fit = gcp.fit(X_train_global, y_train_global)
        gcp_fit_global = gcp_fit

    def ML_prediction(self):
        global gcp_y_predict_global

        gc.collect()
        gcp_y_predict = gcp_fit_global.predict(X_test_global)
        gcp_y_predict_global = gcp_y_predict.round()

    def evaluation(self):

        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(y_test_global, gcp_y_predict_global)
        print("Accuracy:", accuracy)

        from sklearn.metrics import balanced_accuracy_score
        balanced_accuracy_score = balanced_accuracy_score(y_test_global, gcp_y_predict_global)
        print("balanced_accuracy_score:", balanced_accuracy_score)

        from sklearn.metrics import top_k_accuracy_score
        #top_k_accuracy_score = top_k_accuracy_score(y_test_global,gcp_fit_global.predict_proba(X_test_global)[:, 1], k = 1)
        #print("top_k_accuracy_score:", top_k_accuracy_score)

        from sklearn.metrics import average_precision_score
        #average_precision_score = average_precision_score(y_test_global,gcp_fit_global.predict_proba(X_test_global)[:, 1])
        #print("average_precision_score:", average_precision_score)

        from sklearn.metrics import brier_score_loss
        #brier_score_loss = brier_score_loss(y_test_global,gcp_fit_global.predict_proba(X_test_global)[:, 1])
        #print("brier_score_loss:", brier_score_loss)

        from sklearn.metrics import f1_score
        f1_score = f1_score(y_test_global, gcp_y_predict_global, average='macro')
        print("f1_score_macro:", f1_score)
        from sklearn.metrics import f1_score
        f1_score_1 = f1_score(y_test_global, gcp_y_predict_global, average='micro')
        print("f1_score_micro:", f1_score_1)
        f1_score_2 = f1_score(y_test_global, gcp_y_predict_global, average='weighted')
        print("f1_score_weighted:", f1_score_2)
        f1_score_3 = f1_score(y_test_global, gcp_y_predict_global, average=None)
        print("f1_score_None:", f1_score_3)
        #f1_score_4 = f1_score(y_test_global, gcp_y_predict_global, average='binary')
        #print("f1_score_binary:", f1_score_4)

        from sklearn.metrics import log_loss
        #log_loss = log_loss(y_test_global,gcp_fit_global.predict_proba(X_test_global)[:, 1])
        #print("log_loss:", log_loss)

        from sklearn.metrics import precision_score
        precision_score_macro = precision_score(y_test_global, gcp_y_predict_global, average='macro')
        print("precision_score_macro:", precision_score_macro)
        precision_score_micro = precision_score(y_test_global, gcp_y_predict_global, average='micro')
        print("precision_score_micro:", precision_score_micro)
        precision_score_weighted = precision_score(y_test_global, gcp_y_predict_global, average='weighted')
        print("precision_score_weighted:", precision_score_weighted)
        #precision_score_binary = precision_score(y_test_global, gcp_y_predict_global, average='binary')
        #print("precision_score_binary:", precision_score_binary)
        precision_score_None = precision_score(y_test_global, gcp_y_predict_global, average=None)
        print("precision_score_None:", precision_score_None)
        precision_score_None_ZD_1 = precision_score(y_test_global, gcp_y_predict_global, average=None, zero_division=1)
        print("precision_score_None_ZD_1:", precision_score_None_ZD_1)
        precision_score_None_ZD_NAN = precision_score(y_test_global, gcp_y_predict_global, average=None, zero_division=np.nan)
        print("precision_score_None_ZD_NAN:", precision_score_None_ZD_NAN)

        from sklearn.metrics import recall_score
        recall_score_macro = recall_score(y_test_global, gcp_y_predict_global, average='macro')
        print("recall_score_macro:", recall_score_macro)
        recall_score_micro = recall_score(y_test_global, gcp_y_predict_global, average='micro')
        print("recall_score_micro:", recall_score_micro)
        recall_score_weighted = recall_score(y_test_global, gcp_y_predict_global, average='weighted')
        print("recall_score_weighted:", recall_score_weighted)
        #recall_score_binary = recall_score(y_test_global, gcp_y_predict_global, average='binary')
        #print("recall_score_binary:", recall_score_binary)
        recall_score_None = recall_score(y_test_global, gcp_y_predict_global, average=None)
        print("recall_score_None:", recall_score_None)
        recall_score_None_ZD_1 = recall_score(y_test_global, gcp_y_predict_global, average=None, zero_division=1)
        print("recall_score_None_ZD_1:", recall_score_None_ZD_1)
        recall_score_None_ZD_NAN = recall_score(y_test_global, gcp_y_predict_global, average=None, zero_division=np.nan)
        print("recall_score_None_ZD_NAN:", recall_score_None_ZD_NAN)

        from sklearn.metrics import jaccard_score
        jaccard_score_macro = jaccard_score(y_test_global, gcp_y_predict_global, average='macro')
        print("jaccard_score_macro:", jaccard_score_macro)
        jaccard_score_micro = jaccard_score(y_test_global, gcp_y_predict_global, average='micro')
        print("jaccard_score_micro:", jaccard_score_micro)
        jaccard_score_weighted = jaccard_score(y_test_global, gcp_y_predict_global, average='weighted')
        print("jaccard_score_weighted:", jaccard_score_weighted)
        #jaccard_score_binary = jaccard_score(y_test_global, gcp_y_predict_global, average='binary')
        #print("jaccard_score_binary:", jaccard_score_binary)
        jaccard_score_None = jaccard_score(y_test_global, gcp_y_predict_global, average=None)
        print("jaccard_score_None:", jaccard_score_None)

        from sklearn.metrics import roc_auc_score
        #roc_auc_score = roc_auc_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1])
        #print('roc_auc_score:', roc_auc_score)

        from sklearn.metrics import roc_auc_score
        #roc_auc_score_macro = roc_auc_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1], average='macro')
        #print("roc_auc_score_macro:", roc_auc_score_macro)
        #roc_auc_score_micro = roc_auc_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1], average='micro')
        #print("roc_auc_score_micro:", roc_auc_score_micro)
        #roc_auc_score_weighted = roc_auc_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1], 
                                               #average='weighted')
        #print("roc_auc_score_weighted:", roc_auc_score_weighted)
        #roc_auc_score_None = roc_auc_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1], average=None)
        #print("roc_auc_score_None:", roc_auc_score_None)

        from sklearn.metrics import d2_log_loss_score
        #d2_log_loss_score = d2_log_loss_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1],
                                              #sample_weight=None, labels=None)
        #print("d2_log_loss_score:", d2_log_loss_score)

        from sklearn.metrics import d2_brier_score
        #d2_brier_score = d2_brier_score(y_test_global, gcp_fit_global.predict_proba(X_test_global)[:, 1], 
                                        #sample_weight=None, labels=None)
        #print("d2_brier_score:", d2_brier_score)
        

    def main(self):
        gc.collect()
        self.train_split()
        self.ML_fitting()
        self.ML_prediction()
        self.evaluation()
        gc.collect()
        

In [27]:
gcp_1 = LinearSVR_Class(df1, 0.0, 0.0001, 1.0, 
                        'epsilon_insensitive', True, 
                        1.0, 'auto', 0, 
                        None, 1000)
gcp_1.main()

/home/xy/Desktop/ml/scikitlearn/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/xy/Desktop/ml/scikitlearn/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/home/xy/Desktop/ml/scikitlearn/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/xy/Desktop/ml/scikitlearn/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Accuracy: 0.944275
balanced_accuracy_score: 0.9170594679012217
f1_score_macro: 0.46440066413777836
f1_score_micro: 0.944275
f1_score_weighted: 0.9434928697073518
f1_score_None: [0.         0.89555879 0.96204387 0.        ]
precision_score_macro: 0.47139500743581786
precision_score_micro: 0.944275
precision_score_weighted: 0.9442092010899924
precision_score_None: [0.         0.93957882 0.94600121 0.        ]
precision_score_None_ZD_1: [0.         0.93957882 0.94600121 0.        ]
precision_score_None_ZD_NAN: [0.         0.93957882 0.94600121 0.        ]
recall_score_macro: 0.45852973395061086
recall_score_micro: 0.944275
recall_score_weighted: 0.944275
recall_score_None: [0.         0.8554789  0.97864004 0.        ]
recall_score_None_ZD_1: [1.         0.8554789  0.97864004 1.        ]
recall_score_None_ZD_NAN: [       nan 0.8554789  0.97864004        nan]
jaccard_score_macro: 0.4344335498310899
jaccard_score_micro: 0.8944327357976746
jaccard_score_weighted: 0.8944987020436551
jaccard_sc

In [28]:
gcp_fit_global

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('linearsvr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](67,)","['manufacturing_year','battery_capacity_kwh','odometer_km',..., 'Battery_Chemistry_Num_Value','Drive_Type_Num_Value', 'Vehicle_Type_Num_Value']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,67
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [29]:
gcp_y_predict_global

array([1., 1., 1., ..., 1., 1., 0.], shape=(40000,))

In [30]:
np.unique(gcp_y_predict_global)

array([-1., -0.,  1.,  2.])